# Uncertainty Analysis: Monte Carlo vs. Parameter Bounds

Real flow demand and pipe roughness aren't known exactly. This notebook propagates that uncertainty through the hydraulic model to get a *distribution* of head loss / pump power outcomes, rather than a single point estimate.

In [ ]:
import sys
sys.path.insert(0, '..')


from src.simulation.scenario import PipeScenario
from src.simulation.monte_carlo import run_monte_carlo, ParameterUncertainty, summary_statistics
from src.plots.plot_efficiency import monte_carlo_histogram_figure

## Define the baseline scenario and uncertain inputs

- Flow rate: triangular distribution (low/mode/high demand)
- Roughness: uniform distribution (manufacturing/aging variability)

In [ ]:
base = PipeScenario(diameter_m=0.0508, length_m=100.0, flow_rate_m3s=0.0005,
                    fittings={'elbow_90_standard': 4, 'gate_valve_open': 1})

uncertainties = [
    ParameterUncertainty('flow_rate_m3s', 'triangular',
                          {'low': 0.0003, 'mode': 0.0005, 'high': 0.0009}),
    ParameterUncertainty('roughness_m', 'uniform',
                          {'low': 1.0e-6, 'high': 3.0e-6}),
]

## Run the Monte Carlo simulation

In [ ]:
mc_df = run_monte_carlo(base, uncertainties, n_samples=2000, seed=42)
mc_df.head()

## Summary statistics (mean, std, P5/median/P95)

In [ ]:
summary = summary_statistics(
    mc_df, columns=['total_loss_m', 'pressure_drop_Pa', 'shaft_power_W', 'exergy_destroyed_W']
)
summary

## Compare against deterministic (point-estimate) bounds

A naive "worst case / best case" parameter-bound approach would just plug in (low, low) and (high, high) — compare that against the full Monte Carlo distribution's P5-P95 range.

In [ ]:
from src.simulation.scenario import run_simulation

low = run_simulation(diameter_m=0.0508, flow_rate_m3s=0.0003, length_m=100.0,
                      roughness_m=1.0e-6,
                      fittings={'elbow_90_standard': 4, 'gate_valve_open': 1})
high = run_simulation(diameter_m=0.0508, flow_rate_m3s=0.0009, length_m=100.0,
                       roughness_m=3.0e-6,
                       fittings={'elbow_90_standard': 4, 'gate_valve_open': 1})

print(f"Naive bound-stacking range:  {low.head_loss.total_loss_m:.4f} - "
      f"{high.head_loss.total_loss_m:.4f} m")
print(f"Monte Carlo P5-P95 range:    {summary.loc['p05','total_loss_m']:.4f} - "
      f"{summary.loc['p95','total_loss_m']:.4f} m")
print('\nNote: naive bound-stacking (worst-case on every input '
      'simultaneously) is typically far more conservative than the '
      'Monte Carlo P5-P95 band, since uncorrelated uncertain inputs '
      'rarely all land at their extremes together.')

## Visualize the distribution

In [ ]:
monte_carlo_histogram_figure(mc_df, 'total_loss_m', x_label='Total Head Loss (m)').show()

In [ ]:
monte_carlo_histogram_figure(mc_df, 'shaft_power_W', x_label='Pump Shaft Power (W)').show()